In [1]:
import pandas as pd

## 1. Создание датасета. Извлечение времени.

In [2]:
views = pd.read_csv("../data/feed-views.log", delimiter="\t", header=None, names=["datetime","user"])
views['datetime'].dtype

dtype('O')

- приводим к нужному типу

In [3]:
views['datetime'] = pd.to_datetime(views['datetime'])
views['datetime'].dtype

dtype('<M8[ns]')

- извлекаем единицы измерения времени

In [4]:
views['year']   = views['datetime'].dt.year
views['month']  = views['datetime'].dt.month
views['day']    = views['datetime'].dt.day
views['hour']   = views['datetime'].dt.hour
views['minute'] = views['datetime'].dt.minute
views['second'] = views['datetime'].dt.second
views

,datetime,user,year,month,day,hour,minute,second
0,2020-04-17 12:01:08.463179,artem,2020,4,17,12,1,8
1,2020-04-17 12:01:23.743946,artem,2020,4,17,12,1,23
2,2020-04-17 12:27:30.646665,artem,2020,4,17,12,27,30
3,2020-04-17 12:35:44.884757,artem,2020,4,17,12,35,44
4,2020-04-17 12:35:52.735016,artem,2020,4,17,12,35,52
...,...,...,...,...,...,...,...,...
1071,2020-05-21 18:45:20.441142,valentina,2020,5,21,18,45,20
1072,2020-05-21 23:03:06.457819,maxim,2020,5,21,23,3,6
1073,2020-05-21 23:23:49.995349,pavel,2020,5,21,23,23,49
1074,2020-05-21 23:49:22.386789,artem,2020,5,21,23,49,22


## 2. Разбиение по бинам. Смена index.

In [5]:
bins = [0, 4, 7, 11, 17, 20, 24]
labels = ["night", "early morning", "morning", "afternoon", "early evening", "evening"]

views["daytime"] = pd.cut(
    views["hour"],
    bins=bins,
    labels=labels,
    right=False,
    include_lowest=True
)

views.set_index("user", inplace=True)
views

,datetime,year,month,day,hour,minute,second,daytime
user,,,,,,,,
artem,2020-04-17 12:01:08.463179,2020,4,17,12,1,8,afternoon
artem,2020-04-17 12:01:23.743946,2020,4,17,12,1,23,afternoon
artem,2020-04-17 12:27:30.646665,2020,4,17,12,27,30,afternoon
artem,2020-04-17 12:35:44.884757,2020,4,17,12,35,44,afternoon
artem,2020-04-17 12:35:52.735016,2020,4,17,12,35,52,afternoon
...,...,...,...,...,...,...,...,...
valentina,2020-05-21 18:45:20.441142,2020,5,21,18,45,20,early evening
maxim,2020-05-21 23:03:06.457819,2020,5,21,23,3,6,evening
pavel,2020-05-21 23:23:49.995349,2020,5,21,23,23,49,evening


## 3. Некоторые подсчеты.

In [6]:
views.count()

datetime    1076
year        1076
month       1076
day         1076
hour        1076
minute      1076
second      1076
daytime     1076
dtype: int64

In [7]:
views["daytime"].value_counts()

daytime
evening          509
afternoon        252
early evening    145
night            129
morning           36
early morning      5
Name: count, dtype: int64

## 4. Сортировка.

In [8]:
views.sort_values(by=['hour','minute','second'], inplace=True)
views

,datetime,year,month,day,hour,minute,second,daytime
user,,,,,,,,
valentina,2020-05-15 00:00:13.222265,2020,5,15,0,0,13,night
valentina,2020-05-15 00:01:05.153738,2020,5,15,0,1,5,night
pavel,2020-05-12 00:01:27.764025,2020,5,12,0,1,27,night
pavel,2020-05-12 00:01:38.444917,2020,5,12,0,1,38,night
pavel,2020-05-12 00:01:55.395042,2020,5,12,0,1,55,night
...,...,...,...,...,...,...,...,...
artem,2020-05-21 23:49:22.386789,2020,5,21,23,49,22,evening
anatoliy,2020-05-09 23:53:55.599821,2020,5,9,23,53,55,evening
pavel,2020-05-09 23:54:54.260791,2020,5,9,23,54,54,evening


## 5. Сбор статистики.

- самый поздний час ночью

In [9]:
max_night = views.loc[views["daytime"]=="night", "hour"].max()

max_night

np.int32(3)

- самый ранний час утром

In [10]:
min_morning = views.loc[views["daytime"]=="morning", "hour"].min()

min_morning

np.int32(8)

- Кто заходил в тот час ночью и кто утром

In [11]:
night_user = views.loc[views["hour"]==max_night].index.tolist()

night_user

['konstantin', 'konstantin', 'konstantin']

In [12]:
morning_user = views.loc[views["hour"]==min_morning].index.tolist()

morning_user

['alexander', 'alexander']

- Мода времени суток и часов

In [13]:
mode = views["daytime"].mode()
mode

0    evening
Name: daytime, dtype: category
Categories (6, object): ['night' < 'early morning' < 'morning' < 'afternoon' < 'early evening' < 'evening']

In [18]:
hours_mode = views["hour"].mode()
hours_mode

0    22
Name: hour, dtype: int32

## 6. Самые ранние и самые поздние часы за весь день.

In [14]:
views.nsmallest(3,"hour")

,datetime,year,month,day,hour,minute,second,daytime
user,,,,,,,,
valentina,2020-05-15 00:00:13.222265,2020,5,15,0,0,13,night
valentina,2020-05-15 00:01:05.153738,2020,5,15,0,1,5,night
pavel,2020-05-12 00:01:27.764025,2020,5,12,0,1,27,night


In [15]:
views.nlargest(3,"hour")

,datetime,year,month,day,hour,minute,second,daytime
user,,,,,,,,
ekaterina,2020-05-14 23:02:11.327532,2020,5,14,23,2,11,evening
ekaterina,2020-05-14 23:02:14.494985,2020,5,14,23,2,14,evening
ekaterina,2020-05-14 23:02:15.588808,2020,5,14,23,2,15,evening


## 7. Метод describe

In [16]:
d = views.describe()
d

,datetime,year,month,day,hour,minute,second
count,1076,1076.0,1076.000000,1076.000000,1076.000000,1076.000000,1076.000000
mean,2020-05-10 09:00:41.211420672,2020.0,4.870818,13.552974,16.249071,29.629182,29.500929
min,2020-04-17 12:01:08.463179,2020.0,4.000000,1.000000,0.000000,0.000000,0.000000
25%,2020-05-10 01:13:49.857472,2020.0,5.000000,11.000000,13.000000,14.000000,14.000000
50%,2020-05-11 22:48:35.302552832,2020.0,5.000000,13.000000,19.000000,29.000000,30.000000
75%,2020-05-14 14:44:34.749530624,2020.0,5.000000,15.000000,22.000000,46.000000,45.000000
max,2020-05-22 10:36:14.662600,2020.0,5.000000,30.000000,23.000000,59.000000,59.000000
std,NaN,0.0,0.335557,4.906567,6.955490,17.689388,17.405506


- размах по часам и сам интервал

In [17]:
d_h = d['hour']
iqr = d_h['75%'] - d_h['25%']
print(f"{d_h['25%']:.0f}h - {d_h['75%']:.0f}h")
iqr

13h - 22h


np.float64(9.0)